# BERTopic inspection

Counts for the configured corpus filter, the Qdrant embedding collection, discovery topics, and generated headlines.


In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
from IPython.display import display
from qdrant_client import QdrantClient
from sqlalchemy import and_, func, inspect, or_, select, text

from src.common.db import create_processor_engine
from src.common.settings import Settings
from src.corpus.select import _reconstruct_abstract
from src.models.labels import TopicLabel, WorkSummary
from src.models.topics import DiscoveryTopic
from src.models.upstream import work_current, work_versions

TOK_RATIO = 3.0
CONTEXT_LEN = 4096

settings = Settings()
engine = create_processor_engine(
    settings.database_url, upstream_schema=settings.upstream_schema
)
qdrant = QdrantClient(url=settings.qdrant_url)
config = settings.corpus
config = settings.corpus.model_copy(
    update={
        "published_from": date(2026, 1, 1),
        "published_to": date(2026, 1, 31),
    }
)

pd.set_option("display.max_colwidth", 160)


## Works

Selected works use the corpus filter from settings: collection scopes, work types, publication interval, excluded OpenAlex domains, a stored abstract, and no cross-scope version conflict.


In [2]:
LOAD_CACHE_WORKS_DF = True
works_cache_path = Path("data") / "works_df.parquet"

print(
    f"Scopes: {len(config.scope_ids)}. "
    f"\nTypes: {', '.join(config.work_types)}. "
    f"\nPublished: {config.published_from.isoformat()} to {config.published_to.isoformat()}. "
    f"\nExcluded domains: {', '.join(config.excluded_domain_ids) or 'none'}."
)

payload = work_versions.c.payload
work_type = payload["type"].astext
published = payload["publication_date"].astext
domain_id = payload["primary_topic"]["domain"]["id"].astext
abstract_index = payload["abstract_inverted_index"]
abstract_kind = func.jsonb_typeof(abstract_index)
joined = work_current.join(
    work_versions, work_versions.c.id == work_current.c.version_id
)
# Compare the jsonb value. Casting the inverted index to text spills a huge temp file.
selected_filter = [
    work_current.c.scope_id.in_(config.scope_ids),
    work_type.in_(config.work_types),
    published >= config.published_from.isoformat(),
    published <= config.published_to.isoformat(),
    or_(domain_id.is_(None), domain_id.not_in(config.excluded_domain_ids)),
    or_(
        and_(abstract_kind == "object", abstract_index != text("'{}'::jsonb")),
        and_(
            abstract_kind == "string",
            abstract_index != text("""'"{}"'::jsonb"""),
            abstract_index != text("""'"null"'::jsonb"""),
            abstract_index != text("""'""'::jsonb"""),
        ),
    ),
]
# One scope has one current version per work. Several scopes can disagree.
if len(config.scope_ids) > 1:
    versions = func.count(func.distinct(work_current.c.version_id))
    conflicts = (
        select(work_current.c.entity_id)
        .where(work_current.c.scope_id.in_(config.scope_ids))
        .group_by(work_current.c.entity_id)
        .having(versions > 1)
    )
    selected_filter.append(work_current.c.entity_id.not_in(conflicts))

title_text = payload["title"].astext
title_length = func.length(func.trim(title_text))
selected = (
    select(
        work_type.label("type"),
        title_length.label("title_length"),
        abstract_index.label("abstract_index"),
    )
    .select_from(joined)
    .where(*selected_filter)
)

if LOAD_CACHE_WORKS_DF and works_cache_path.is_file():
    works_df = pd.read_parquet(works_cache_path)
    works_by_type = (
        works_df["type"]
        .value_counts()
        .rename_axis("type")
        .reset_index(name="works")
    )
    print(f"Loaded works from {works_cache_path}")
elif not config.scope_ids:
    works_df = pd.DataFrame(columns=["type", "title_length", "abstract_length"])
    works_by_type = pd.DataFrame(columns=["type", "works"])
    print("No collection scopes are configured. Set PWF_CORPUS__SCOPE_IDS.")
else:
    with engine.connect() as connection:
        # Parallel workers spill this scan to disk and can fill the data volume.
        started = time.perf_counter()
        connection.execute(text("SET max_parallel_workers_per_gather = 0"))
        works_df = pd.read_sql(selected, connection)
        elapsed = time.perf_counter() - started
    print(f"Query duration: {elapsed:.2f} s")
    works_df["title_length"] = works_df["title_length"].fillna(0).astype(int)
    works_df["abstract_length"] = [
        len(_reconstruct_abstract(idx) or "")
        for idx in works_df["abstract_index"]
    ]
    # Parquet cannot store the inverted-index dicts. Keep them as JSON text.
    works_df["abstract_index"] = [
        value if isinstance(value, str) else json.dumps(value)
        for value in works_df["abstract_index"]
    ]
    works_by_type = (
        works_df["type"]
        .value_counts()
        .rename_axis("type")
        .reset_index(name="works")
    )
    works_cache_path.parent.mkdir(parents=True, exist_ok=True)
    works_df.to_parquet(works_cache_path, index=False)
    print(f"Saved works to {works_cache_path}")

selected_works = len(works_df)
print(f"Selected works: {selected_works}")
display(works_by_type)

if len(works_df):
    figure = px.bar(
        works_by_type,
        x="type",
        y="works",
        title="Selected works by type",
        text="works",
    )
    figure.update_layout(xaxis_title="Work type", yaxis_title="Works")
    figure.show()

    figure_title = px.histogram(
        works_df,
        x="title_length",
        title="Works count by title length (chars)",
    )
    figure_title.update_layout(
        xaxis_title="Title length (chars)",
        yaxis_title="Works count",
    )
    figure_title.show()

    figure_abstract = px.histogram(
        works_df,
        x="abstract_length",
        title="Works count by abstract length (chars)",
    )
    figure_abstract.update_layout(
        xaxis_title="Abstract length (chars)",
        yaxis_title="Works count",
    )
    figure_abstract.show()

    works_df["title_context"] = works_df["title_length"] / TOK_RATIO / CONTEXT_LEN
    works_df["abstract_context"] = (
        works_df["abstract_length"] / TOK_RATIO / CONTEXT_LEN
    )

    figure_title_context = px.histogram(
        works_df,
        x="title_context",
        title="Titles count by context window size",
    )
    figure_title_context.update_layout(
        xaxis_title="Context window size",
        yaxis_title="Titles count",
    )
    figure_title_context.show()

    figure_abstract_context = px.histogram(
        works_df,
        x="abstract_context",
        title="Abstract count by context window size",
    )
    figure_abstract_context.update_layout(
        xaxis_title="Context window size",
        yaxis_title="Abstract count",
    )
    figure_abstract_context.show()


Scopes: 1. 
Types: article, preprint, conference-paper. 
Published: 2026-01-01 to 2026-01-31. 
Excluded domains: https://openalex.org/domains/2.
Loaded works from data/works_df.parquet
Selected works: 38534


,type,works
0,article,33590
1,conference-paper,2880
2,preprint,2064


In [3]:
works_df.head(3)

,type,title_length,abstract_index,abstract_length,title_context,abstract_context
0,conference-paper,154,"{""The"": [0, 74, 102], ""modern"": [1, 67], ""agricultural"": [2, 147], ""sector"": [3, 148], ""is"": [4, 107], ""under"": [5], ""intense"": [6], ""pressure"": [7], ""towar...",1195,0.012533,0.097249
1,article,145,"{""Background"": [0], ""Coexisting"": [1], ""forms"": [2, 29, 289], ""of"": [3, 30, 70, 90, 113, 143, 149, 155, 163, 244, 252, 264, 272, 290, 301, 310, 360, 367], ""...",2786,0.011800,0.226725
2,article,229,"{""Background"": [0], ""and"": [1, 30, 43, 78, 95, 111, 145, 191, 198, 221, 230, 237], ""Purpose"": [2], ""Glioblastoma,"": [3], ""the"": [4, 46, 170, 180], ""most"": [...",1796,0.018636,0.146159


## Embeddings

Point count in the configured Qdrant collection, plus one point's payload and vector attributes.


In [4]:
collection = settings.qdrant_collection
print(f"Qdrant: {settings.qdrant_url}")
print(f"Collection: {collection}")

if not qdrant.collection_exists(collection):
    print("Collection is not present. Embedding count: 0")
else:
    info = qdrant.get_collection(collection)
    embedding_count = qdrant.count(collection, exact=True).count
    print(f"Embeddings: {embedding_count}")
    vectors = info.config.params.vectors
    print(f"Vector params: {vectors}")
    points, _next_offset = qdrant.scroll(
        collection_name=collection,
        limit=1,
        with_payload=True,
        with_vectors=True,
    )
    if not points:
        print("Collection has no points.")
    else:
        point = points[0]
        payload_frame = pd.DataFrame(
            [{"field": key, "value": value} for key, value in sorted((point.payload or {}).items())]
        )
        vector = point.vector
        if isinstance(vector, dict):
            vector = next(iter(vector.values()))
        print(f"Point id: {point.id}")
        print(f"Vector dimensions: {len(vector) if vector is not None else 0}")
        display(payload_frame)
        # if vector:
        #     vector_frame = pd.DataFrame({"dimension": range(len(vector)), "value": vector})
        #     figure = px.line(
        #         vector_frame,
        #         x="dimension",
        #         y="value",
        #         title=f"Embedding vector {point.id}",
        #     )
        #     figure.update_layout(xaxis_title="Dimension", yaxis_title="Value")
        #     figure.show()


Qdrant: http://127.0.0.1:16333
Collection: openalex_tabstract


Embeddings: 38030
Vector params: size=1024 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None
Point id: 00005099-6235-5b34-bcdb-27da35f18be7
Vector dimensions: 1024


,field,value
0,language,en
1,model,microsoft/harrier-oss-v1-0.6b
2,model_revision,main
3,primary_domain,https://openalex.org/domains/3
4,publication_date,2026-01-01
5,text_hash,f3d5e92d6a9479b589379c052c150704b4ad906c7f4fef77d8a8e90553dae6be
6,work_id,https://openalex.org/W7125911450
7,work_version_id,c48fefb3ef9f83b232cdf6df1adf1e4a6f06fbdf45abaa5678b022bd442c2e7d


## Topics and clusters

Discovery topics stored in `pwf_discovery_topics`, then the five largest and five smallest by member-work count.


In [5]:
def _keyword_terms(keywords: object, limit: int = 8) -> str:
    if not isinstance(keywords, list):
        return ""
    terms = [str(item["term"]) for item in keywords[:limit] if isinstance(item, dict) and "term" in item]
    return ", ".join(terms)

inspector = inspect(engine)
if not inspector.has_table(DiscoveryTopic.__tablename__):
    print(f"{DiscoveryTopic.__tablename__} is not in this database.")
    print("Discovery topics: unavailable")
else:
    topic_count = select(func.count()).select_from(DiscoveryTopic)
    largest = (
        select(
            DiscoveryTopic.discovery_topic_id,
            DiscoveryTopic.topic_run_id,
            DiscoveryTopic.size,
            DiscoveryTopic.keywords,
        )
        .order_by(DiscoveryTopic.size.desc(), DiscoveryTopic.discovery_topic_id)
        .limit(5)
    )
    smallest = (
        select(
            DiscoveryTopic.discovery_topic_id,
            DiscoveryTopic.topic_run_id,
            DiscoveryTopic.size,
            DiscoveryTopic.keywords,
        )
        .order_by(DiscoveryTopic.size.asc(), DiscoveryTopic.discovery_topic_id)
        .limit(5)
    )
    with engine.connect() as connection:
        topic_total = connection.execute(topic_count).scalar_one()
        largest_frame = pd.DataFrame(connection.execute(largest).mappings().all())
        smallest_frame = pd.DataFrame(connection.execute(smallest).mappings().all())
    print(f"Discovery topics: {topic_total}")
    for frame in (largest_frame, smallest_frame):
        if len(frame):
            frame["keywords"] = frame["keywords"].map(_keyword_terms)
    print("Five largest")
    display(largest_frame)
    print("Five smallest")
    display(smallest_frame)
    if len(largest_frame) or len(smallest_frame):
        largest_frame = largest_frame.assign(rank="largest")
        smallest_frame = smallest_frame.assign(rank="smallest")
        ranked = pd.concat([largest_frame, smallest_frame], ignore_index=True)
        ranked["label"] = ranked["discovery_topic_id"].astype(str).str.slice(0, 8)
        figure = px.bar(
            ranked,
            x="label",
            y="size",
            color="rank",
            title="Largest and smallest discovery topics",
            hover_data=["keywords"],
        )
        figure.update_layout(xaxis_title="Discovery topic", yaxis_title="Member works")
        figure.show()


Discovery topics: 280
Five largest


,discovery_topic_id,topic_run_id,size,keywords
0,2f12a0f2-badc-4e2f-9462-1b9a2df6dabc,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,588,"gait, motor, stimulation, eeg, neural, learning, training, connectivity"
1,5264b5d8-009a-474e-ad98-ea5981a46d78,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,423,"sensory, content, fermentation, protein, flour, extraction, food, antioxidant"
2,04809f3c-f023-473e-911e-563598605900,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,416,"aortic, patients, artery, valve, atrial, coronary, ventricular, left"
3,421d7235-fa58-43aa-846c-2f986f56db96,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,413,"cells, cancer, tumor, cell, expression, immune, therapeutic, progression"
4,5e5fef58-1e55-4760-9c23-3dbd64816088,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,402,"molecular, docking, compounds, binding, derivatives, drug, inhibitors, discovery"


Five smallest


,discovery_topic_id,topic_run_id,size,keywords
0,12d36a60-8205-4deb-a6ad-51cbcc7a9765,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,15,"lusophone, eat, fat, pca, weight, scd, cardiovascular, utilizers"
1,4b84f76a-b656-4ab0-bfe6-3ceeb2704692,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,15,"co, mgo, assimilating, methane, alcoholassisted, nonco, conversion, carbon"
2,5d4811f5-7573-4c7e-a3a3-60ed876db8cd,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,15,"routing, vlsi, placement, congestion, design, slicing, fpgas, chip"
3,cb4ca449-f3e6-4969-a4d8-4ca4b6541add,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,15,"llm, jaas, dualbreach, prompts, guardrails, haste, code, llmbased"
4,ff0d1f3d-c96c-4026-8faa-5bc178055ef6,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,15,"antifungal, albicans, nora, isoflavonoids, antibiofilm, mfc, undecalactone, prenylated"


## Headlines

Work summaries in `pwf_work_summaries`, headlines in `pwf_topic_labels`, then the five longest and five shortest headlines.


In [6]:
if not inspector.has_table(WorkSummary.__tablename__) or not inspector.has_table(
    TopicLabel.__tablename__
):
    print("Summary or headline tables are not in this database.")
    print("Summaries: unavailable")
    print("Headlines: unavailable")
else:
    summary_count = select(func.count()).select_from(WorkSummary)
    headline_count = (
        select(func.count())
        .select_from(TopicLabel)
        .where(TopicLabel.headline.is_not(None))
    )
    headline_length = func.length(TopicLabel.headline)
    columns = (
        TopicLabel.discovery_topic_id,
        TopicLabel.topic_run_id,
        TopicLabel.headline,
        headline_length.label("characters"),
        TopicLabel.status,
    )
    longest = (
        select(*columns)
        .where(TopicLabel.headline.is_not(None))
        .order_by(headline_length.desc(), TopicLabel.discovery_topic_id)
        .limit(5)
    )
    shortest = (
        select(*columns)
        .where(TopicLabel.headline.is_not(None))
        .order_by(headline_length.asc(), TopicLabel.discovery_topic_id)
        .limit(5)
    )
    with engine.connect() as connection:
        summaries = connection.execute(summary_count).scalar_one()
        headlines = connection.execute(headline_count).scalar_one()
        longest_frame = pd.DataFrame(connection.execute(longest).mappings().all())
        shortest_frame = pd.DataFrame(connection.execute(shortest).mappings().all())
    print(f"Summaries: {summaries}")
    print(f"Headlines: {headlines}")
    print("Five largest")
    display(longest_frame)
    print("Five smallest")
    display(shortest_frame)
    if len(longest_frame) or len(shortest_frame):
        longest_frame = longest_frame.assign(rank="largest")
        shortest_frame = shortest_frame.assign(rank="smallest")
        ranked = pd.concat([longest_frame, shortest_frame], ignore_index=True)
        ranked["label"] = ranked["discovery_topic_id"].astype(str).str.slice(0, 8)
        figure = px.bar(
            ranked,
            x="label",
            y="characters",
            color="rank",
            title="Largest and smallest headlines",
            hover_data=["headline"],
        )
        figure.update_layout(xaxis_title="Discovery topic", yaxis_title="Headline characters")
        figure.show()


Summaries: 1159
Headlines: 75
Five largest


,discovery_topic_id,topic_run_id,headline,characters,status
0,095ea347-a5ce-4065-b52e-81f21a0d0ef4,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Next-Generation Retrieval-Augmented Generation: Multi-Agent Frameworks, Multimodal Reasoning, and Adaptive Context for Domain-Specific AI",137,succeeded
1,0a314067-d488-40ab-ab1f-dee3b52a11ac,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Sustainable Aquaculture Nutrition: Alternative Ingredients and Functional Additives Enhance Growth, Immunity, and Metabolic Health.",131,succeeded
2,0df0d02d-74bb-4698-933b-424137ce573b,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Dental Material Innovations: Bonding Efficacy, Biocompatibility, and Mechanical Performance in Restorative and Implant Dentistry",128,succeeded
3,2be42316-25b0-4719-b11d-678f9756d149,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Optimizing TB and HIV Management: Prevention, Drug Resistance, and Implementation Strategies in Low- and Middle-Income Countries",128,succeeded
4,351d0634-88ad-46ff-b4ac-55ce41145896,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Development and Validation of Green, Stability-Indicating Analytical Methods for Pharmaceutical Quality Control and Bioanalysis.",128,succeeded


Five smallest


,discovery_topic_id,topic_run_id,headline,characters,status
0,2f200673-1b59-427a-b0e2-1675cfaa8195,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,Rare Neoplasms: Diagnostic Challenges and Management Strategies,63,succeeded
1,0278c702-3484-4a9e-bf66-9220652b50cb,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,Integrative Systematics and Taxonomic Advances Across Diverse Taxa,66,succeeded
2,1c5a38d2-4556-429a-8122-1732b7a4bbcd,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Discourse, Identity, and Islamic Law in Arabic and Islamic Studies",66,succeeded
3,12d36a60-8205-4deb-a6ad-51cbcc7a9765,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,"Cardiometabolic Risk, Chronic Disease Management, and Health Equity",67,succeeded
4,363415a6-2ca0-4717-bd43-e56cdfb5add9,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,Genetic Causality and Immune Biomarkers in Disease Risk and Outcomes,68,succeeded
